# 05 — Metric and target robustness

Reproduces the §3.3 Spearman-convergence numbers and the Appendix B alternative-target numbers.

**Manuscript references:**
- §3.3 (third paragraph): Spearman vs Pearson sign-flips and dispersion, side by side with winsorisation.
- §3.3 (fourth paragraph): RPE1 leverage-specificity statement, pointing to Appendix B.
- Appendix B: sign-flip counts under leverage, DEG count, and log-DEG targets.

**Sources:**
- `precomputed/eval/diag_spearman_n100_summary.csv`
- `precomputed/eval/diag_winsorise_n100_summary.csv` (for side-by-side comparison)
- `precomputed/eval/diag_alttargets_n100_summary.csv`
- `precomputed/eval/diag_spearman_n100.csv` and `diag_alttargets_n100.csv` (400-row per-seed detail, available for further analysis)

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from perturb_style import PRECOMPUTED_DIR

## Load summaries

In [2]:
spear = pd.read_csv(PRECOMPUTED_DIR / "eval" / "diag_spearman_n100_summary.csv")
wins  = pd.read_csv(PRECOMPUTED_DIR / "eval" / "diag_winsorise_n100_summary.csv")
alt   = pd.read_csv(PRECOMPUTED_DIR / "eval" / "diag_alttargets_n100_summary.csv")
print("Spearman summary:")
print(spear.to_string(index=False))
print()
print("Winsorisation summary (for comparison):")
print(wins[["condition", "raw_median", "wins_median", "raw_sign_flips",
             "wins_sign_flips", "raw_MAD", "wins_MAD", "MAD_pct_change",
             "range_pct_change"]].to_string(index=False))

Spearman summary:
      condition  n_present  pearson_median  spearman_median  pearson_range  spearman_range  range_pct_change  pearson_MAD  spearman_MAD  MAD_pct_change  pearson_p5_p95_width  spearman_p5_p95_width  p5_p95_pct_change  pearson_sign_flips  spearman_sign_flips
    K562 random        100          0.0604           0.0899         1.0943          0.7982             -27.1       0.1742        0.1014           -41.8                0.7631                 0.5139              -32.7                  42                   23
K562 stratified        100          0.1006           0.0863         1.1840          0.9477             -20.0       0.1516        0.1039           -31.5                0.7688                 0.5970              -22.3                  40                   30
    RPE1 random        100          0.2898           0.2890         0.7552          0.7880               4.3       0.1001        0.1279            27.8                0.4732                 0.5452               

## §3.3 manuscript numbers, side by side

The convergence is the headline: for each condition, the Spearman sign-flip count and the winsorised sign-flip count land in the same neighbourhood, while the raw Pearson sign-flip count differs sharply for K562. Same story in dispersion: K562 MAD drops by 30-40 % under either robust intervention; RPE1 MAD barely moves.

In [3]:
rows = []
for cond in ["K562 random", "K562 stratified", "RPE1 random", "RPE1 stratified"]:
    s = spear[spear.condition == cond].iloc[0]
    w = wins[wins.condition == cond].iloc[0]
    rows.append({
        "condition": cond,
        "pearson_sign_flips":   f"{int(w.raw_sign_flips)}/100",
        "winsorised_sign_flips": f"{int(w.wins_sign_flips)}/100",
        "spearman_sign_flips":   f"{int(s.spearman_sign_flips)}/100",
        "pearson_MAD":   round(float(w.raw_MAD), 3),
        "winsorised_MAD": round(float(w.wins_MAD), 3),
        "spearman_MAD":   round(float(s.spearman_MAD), 3),
        "pearson_median":   round(float(w.raw_median), 3),
        "spearman_median":  round(float(s.spearman_median), 3),
    })
panel = pd.DataFrame(rows)
print(panel.to_string(index=False))

      condition pearson_sign_flips winsorised_sign_flips spearman_sign_flips  pearson_MAD  winsorised_MAD  spearman_MAD  pearson_median  spearman_median
    K562 random             42/100                22/100              23/100        0.174           0.137         0.101           0.060            0.090
K562 stratified             40/100                25/100              30/100        0.152           0.126         0.104           0.101            0.086
    RPE1 random              5/100                 5/100               6/100        0.100           0.110         0.128           0.290            0.289
RPE1 stratified              8/100                10/100               8/100        0.112           0.112         0.094           0.263            0.264


## Appendix B: sign-flip counts across alternative severity targets

Same predictions, three different severity targets: leverage (canonical, headline section), DEG count, and log(1 + DEG count). Knockdown is omitted from the manuscript table because of a heavy point mass at the -1 sentinel value, but the per-seed knockdown numbers are present in `diag_alttargets_n100.csv` for completeness.

The cell-type contrast collapses when the severity target is changed: RPE1's apparent stability under leverage (5-8/100 sign-flips) does not extend to DEG-based targets (30-49/100), while K562 remains unstable across all three targets (33-56/100).

In [4]:
rows = []
for cond in ["K562 random", "K562 stratified", "RPE1 random", "RPE1 stratified"]:
    a = alt[alt.condition == cond].iloc[0]
    def fmt(t):
        return f"{int(a[f'pearson_{t}_sign_flips'])} / {int(a[f'spearman_{t}_sign_flips'])}"
    rows.append({
        "condition": cond,
        "Leverage P/S": fmt("leverage"),
        "DEG P/S":      fmt("deg"),
        "log-DEG P/S":  fmt("logdeg"),
        "abs(KD) P/S":  fmt("abskd") + "  (exploratory; -1 sentinel"
                                       f" median {int(a['median_n_kd_sentinel'])}/30)",
    })
appendix_b = pd.DataFrame(rows)
print(appendix_b.to_string(index=False))

      condition Leverage P/S DEG P/S log-DEG P/S                                     abs(KD) P/S
    K562 random      42 / 23 54 / 42     39 / 42 38 / 49  (exploratory; -1 sentinel median 1/30)
K562 stratified      40 / 30 56 / 43     35 / 43 33 / 44  (exploratory; -1 sentinel median 1/30)
    RPE1 random        5 / 6 31 / 35     44 / 35 57 / 54  (exploratory; -1 sentinel median 4/30)
RPE1 stratified        8 / 8 34 / 30     49 / 30 47 / 50  (exploratory; -1 sentinel median 4/30)


## Interpretation (manuscript-facing)

**§3.3 Spearman convergence:**
- K562 random: Pearson 42/100 sign-flips, Spearman 23/100, winsorised 22/100.
- K562 stratified: Pearson 40/100, Spearman 30/100, winsorised 25/100.
- RPE1 random: Pearson 5/100, Spearman 6/100, winsorised 5/100.
- RPE1 stratified: Pearson 8/100, Spearman 8/100, winsorised 10/100.

**Appendix B alternative targets:**
- RPE1 random: 5 (leverage Pearson) -> 31 (DEG Pearson) -> 44 (log-DEG Pearson).
- RPE1 stratified: 8 -> 34 -> 49.
- K562 random: 42 -> 54 -> 39.
- K562 stratified: 40 -> 56 -> 35.

Two independent robust interventions on the metric converge on the same K562/RPE1 mechanism distinction. A change of severity target, however, collapses the contrast: RPE1 stability is leverage-specific.